In [1]:
# %% [markdown]
# # STEP 1: Overview
# # Description:
# # This script processes heat load distribution ONLY for explicitly defined model types.
# #
# # Key behaviour:
# # 1. Loops through all building folders
# # 2. ONLY selects model folders matching VALID_MODEL_NAMES
# # 3. Reads heat_load.csv from cfg
# # 4. Combines with HEATING system data
# # 5. Outputs heat_load_dis.csv
# #
# # This ensures strict control and avoids accidental folder selection.

# %%
# STEP 1: Setup paths and imports

import os
import pandas as pd

BASE_MODELS_DIR = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

print("Setup complete.")

Setup complete.


In [2]:
# %%
# STEP 2: Define valid model folders
# Description: Only these model types will be processed

VALID_MODEL_NAMES = {
    "SemiDetached_2F_fab",
    "Detached_2F_fab"
}

print("Valid model names:", VALID_MODEL_NAMES)

Valid model names: {'Detached_2F_fab', 'SemiDetached_2F_fab'}


In [3]:
# %%
# STEP 3: Discover model paths
# Description: Explicitly find valid model folders inside each building

model_map = []  # list of tuples: (building_id, model_path)

for building in os.listdir(BASE_MODELS_DIR):
    building_path = os.path.join(BASE_MODELS_DIR, building)

    if not os.path.isdir(building_path):
        continue

    for sub in os.listdir(building_path):
        sub_path = os.path.join(building_path, sub)

        if sub in VALID_MODEL_NAMES and os.path.isdir(sub_path):
            model_map.append((building, sub_path))

print(f"Found {len(model_map)} valid model(s)")

Found 69 valid model(s)


In [4]:
# %%
# STEP 4: Process single model
# Description: Processes one building + model pair

def process_model(building_id, model_path):
    
    print("\n" + "="*60)
    print(f"Processing: {building_id} | {model_path}")
    
    # Heat load file
    heat_load_path = os.path.join(model_path, "cfg", "heat_load.csv")
    if not os.path.exists(heat_load_path):
        print("Missing heat_load.csv → skipping")
        return
    
    # HEATING folder
    building_path = os.path.join(BASE_MODELS_DIR, building_id)
    heating_folder = os.path.join(building_path, "HEATING")
    
    if not os.path.isdir(heating_folder):
        print("Missing HEATING folder → skipping")
        return
    
    # Load heat load data
    heat_df = pd.read_csv(heat_load_path)
    
    # Total load
    heat_df["total_load"] = (
        heat_df["flr0_heat_load"] +
        heat_df["flr1_heat_load"] +
        heat_df["attic_heat_load"]
    )
    
    # Main efficiency
    main_eff_path = os.path.join(heating_folder, "main_heat_effiency.csv")
    if not os.path.exists(main_eff_path):
        print("Missing main_heat_effiency.csv → skipping")
        return
    
    main_eff_df = pd.read_csv(main_eff_path)
    
    # Secondary system check
    second_eff_path = os.path.join(heating_folder, "second_heat_effiency.csv")
    secondary_exists = os.path.exists(second_eff_path)
    
    if secondary_exists:
        second_eff_df = pd.read_csv(second_eff_path)
        activation_path = os.path.join(heating_folder, "secondary_heating_activation.csv")
        
        if not os.path.exists(activation_path):
            print("Missing activation file → skipping")
            return
        
        activation_df = pd.read_csv(activation_path)
        
        heat_df["secondary_heat_load"] = heat_df["total_load"] * activation_df["Secondary_Heating_Percent"]
        heat_df["main_heat_load"] = heat_df["total_load"] * activation_df["Primary_Heating_Percent"]
        
        heat_df["secondary_heat_power"] = (
            heat_df["secondary_heat_load"] / second_eff_df["second_heat_efficiency"]
        )
    else:
        heat_df["secondary_heat_load"] = 0
        heat_df["main_heat_load"] = heat_df["total_load"]
        heat_df["secondary_heat_power"] = 0
    
    # Main heat power
    heat_df["main_heat_power"] = (
        heat_df["main_heat_load"] / main_eff_df["main_heat_efficiency"]
    )
    
    # Output
    output_df = pd.DataFrame({
        "Time": heat_df["Time"],
        "total_load": heat_df["total_load"],
        "secondary_heat_load": heat_df["secondary_heat_load"],
        "main_heat_load": heat_df["main_heat_load"],
        "secondary_heat_power": heat_df["secondary_heat_power"],
        "main_heat_power": heat_df["main_heat_power"]
    })
    
    output_path = os.path.join(heating_folder, "heat_load_dis_fab.csv")
    output_df.to_csv(output_path, index=False)
    
    print("Processed successfully.")

In [5]:
# %%
# STEP 5: Process all valid models
# Description: Loop through filtered model_map

processed = 0

for building_id, model_path in model_map:
    process_model(building_id, model_path)
    processed += 1

print("\nProcessing complete")
print(f"Total processed: {processed}")


Processing: 1001664924 | /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F_fab
Processed successfully.

Processing: 1001829085 | /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829085/SemiDetached_2F_fab
Processed successfully.

Processing: 1001063637 | /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063637/Detached_2F_fab
Processed successfully.

Processing: 1001664941 | /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664941/Detached_2F_fab
Processed successfully.

Processing: 1235057414 | /Users/jakegehrung/Desktop/data_raw/baseline_models/1235057414/Detached_2F_fab
Processed successfully.

Processing: 1001829079 | /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829079/SemiDetached_2F_fab
Processed successfully.

Processing: 1001664925 | /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664925/Detached_2F_fab
Processed successfully.

Processing: 1000238907 | /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238

In [6]:
# %%
# STEP 6: Validation
# Description: Preview one output file

if model_map:
    test_building = model_map[0][0]
    test_path = os.path.join(BASE_MODELS_DIR, test_building, "HEATING", "heat_load_dis.csv")

    if os.path.exists(test_path):
        df = pd.read_csv(test_path)
        print(df.head())
    else:
        print("No output file found for test building.")
else:
    print("No models were processed.")

       Time  total_load  secondary_heat_load  main_heat_load  \
0  00:30:00        2300                    0            2300   
1  01:00:00        2220                    0            2220   
2  01:30:00        2170                    0            2170   
3  02:00:00        2160                    0            2160   
4  02:30:00        2170                    0            2170   

   secondary_heat_power  main_heat_power  
0                     0       569.508318  
1                     0       541.652157  
2                     0       525.605553  
3                     0       523.183407  
4                     0       526.973129  
